In [9]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
import os

# Data folder path
DATA_PATH = r"C:\Users\HP\OneDrive\Desktop\Har Project\data\UCI_HAR_Dataset"

print("✅ Libraries loaded successfully!")


C:\Users\HP\anaconda3\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


✅ Libraries loaded successfully!


In [11]:
# Load training data
X_train = np.loadtxt(os.path.join(DATA_PATH, 'train', 'X_train.txt'))
y_train = np.loadtxt(os.path.join(DATA_PATH, 'train', 'y_train.txt')).astype(int)

# Load test data
X_test = np.loadtxt(os.path.join(DATA_PATH, 'test', 'X_test.txt'))
y_test = np.loadtxt(os.path.join(DATA_PATH, 'test', 'y_test.txt')).astype(int)

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)

Training data shape: (7352, 561)
Test data shape: (2947, 561)


In [13]:
# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# One-hot encode labels (Activities = 6 classes)
y_train = to_categorical(y_train - 1)
y_test = to_categorical(y_test - 1)

print("✅ Data normalized and labels encoded!")

✅ Data normalized and labels encoded!


In [15]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(6, activation='softmax')  # 6 classes
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

C:\Users\HP\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │          71,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 6)                   │             390 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 80,582 (314.77 KB)

 Trainable params: 80,582 (314.77 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)


Epoch 1/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8211 - loss: 0.4417 - val_accuracy: 0.9267 - val_loss: 0.1856
Epoch 2/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9339 - loss: 0.1693 - val_accuracy: 0.9501 - val_loss: 0.1383
Epoch 3/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9508 - loss: 0.1280 - val_accuracy: 0.9294 - val_loss: 0.1920
Epoch 4/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9555 - loss: 0.1103 - val_accuracy: 0.9138 - val_loss: 0.2418
Epoch 5/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9626 - loss: 0.1028 - val_accuracy: 0.9403 - val_loss: 0.1802
Epoch 6/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9626 - loss: 0.0987 - val_accuracy: 0.9379 - val_loss: 0.2050
Epoch 7/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9642 - loss: 0.0978 - val_accuracy: 0.9192 - val_loss: 0.2954
Epoch 8/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9698 - loss: 0.0825 - val_accuracy: 0.

In [19]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test Accuracy: {accuracy * 100:.2f}%")

✅ Test Accuracy: 95.15%


In [21]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = os.path.join(DATA_PATH, "har_model.tflite")
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"✅ Model saved successfully at: {tflite_path}")


INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmpab6nvlm2\assets


INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmpab6nvlm2\assets


Saved artifact at 'C:\Users\HP\AppData\Local\Temp\tmpab6nvlm2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 561), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2078241783248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2078241783632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2078241782864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2078259839824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2078259840400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2078259841168: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ Model saved successfully at: C:\Users\HP\OneDrive\Desktop\Har Project\data\UCI_HAR_Dataset\har_model.tflite


In [23]:
model.save(os.path.join(DATA_PATH, "har_model.h5"))
